In [ ]:
"""
NOTE:
The scripts contain local absolute paths used during thesis experiments.
To rerun the code, adapt the path definitions
(e.g., IMG_DIR, COCO_JSON, BASE_OUT_DIR) to match your local directory structure.
"""

import json, os
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from datetime import datetime
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

# ----------------------
# Matplotlib defaults
# ----------------------
plt.rcParams.update({
    "figure.dpi": 300,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.6
})

IMG_DIR = "path/to/images"
LABEL_DIR = "path/to/labels"

In [ ]:
# -------------
# Figure 28
# -------------

import json
from collections import Counter

import matplotlib.pyplot as plt


# ---- CONFIG ----
JSON_PATH = "path\to\json"
OUTPUT_PATH = r"output\path\fig28.png"
SMOKE_COLOR = "#ff9999"
NONSMOKE_COLOR = "#66b3ff"
# ----------------


def load_class_distribution(json_path: str) -> tuple[int, int]:
    """Return smoke vs non-smoke image counts."""
    with open(json_path, "r") as f:
        coco = json.load(f)

    ann_counts = Counter(ann["image_id"] for ann in coco["annotations"])
    img_ids = [img["id"] for img in coco["images"]]

    smoke = sum(1 for img_id in img_ids if ann_counts.get(img_id, 0) > 0)
    non_smoke = len(img_ids) - smoke

    return smoke, non_smoke


def plot_pie(smoke: int, non_smoke: int, out_path: str) -> None:
    """Save a pie chart."""
    labels = ["Smoke", "Non-Smoke"]
    values = [smoke, non_smoke]
    colors = [SMOKE_COLOR, NONSMOKE_COLOR]

    plt.figure(figsize=(6, 6))

    wedges, texts, autotexts = plt.pie(
        values,
        labels=None,
        colors=colors,
        autopct="%1.1f%%",
        pctdistance=0.8,
        startangle=90,
        textprops={"fontsize": 12},
    )


    plt.legend(
        wedges,
        labels,
        title="Classes",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.05),
        fontsize=12,
        title_fontsize=12,
        frameon=False,
    )

    plt.title("Class Distribution in Pyro-SDIS Dataset", fontsize=14)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved pie chart → {out_path}")


# ---- RUN ----
if __name__ == "__main__":
    smoke, non_smoke = load_class_distribution(JSON_PATH)
    print(f"Smoke: {smoke}, Non-Smoke: {non_smoke}")

    plot_pie(smoke, non_smoke, OUTPUT_PATH)


Smoke: 28137, Non-Smoke: 5499
Saved pie chart → D:\temp\class_distribution_pie.png


In [ ]:
# ---------------------------------------
# Pyro-SDIS event summaries for Table 10
# ---------------------------------------

import csv
import json
import os
from collections import defaultdict
from datetime import datetime
from typing import Dict


def parse_event_id(filename: str) -> str:
    """
    Extract an event ID from a Pyro-SDIS filename.
    """
    parts = filename.split("_")
    return parts[0] + "_" + parts[1]


def parse_timestamp_from_filename(filename: str) -> datetime | None:
    """
    Extract timestamp from filename if present.
    """
    try:
        ts_part = filename.split("_")[2].replace(".jpg", "")
        return datetime.strptime(ts_part, "%Y-%m-%dT%H-%M-%S")
    except Exception:
        return None


def collect_event_statistics(img_dir: str, lbl_dir: str) -> Dict[str, Dict]:
    """
    Collect per-event statistics:
        - number of frames
        - smoke / no-smoke counts
        - smoke ratio
        - optional timestamps (first/last)
    """
    events = defaultdict(
        lambda: {
            "smoke": 0,
            "nosmoke": 0,
            "images": [],
            "timestamps": [],
        }
    )

    for fname in os.listdir(img_dir):
        if not fname.endswith(".jpg"):
            continue

        event_id = parse_event_id(fname)
        label_path = os.path.join(lbl_dir, fname.replace(".jpg", ".txt"))

        # Read annotation file
        with open(label_path, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            events[event_id]["nosmoke"] += 1
        else:
            events[event_id]["smoke"] += 1

        events[event_id]["images"].append(fname)

        # Extract timestamp
        ts = parse_timestamp_from_filename(fname)
        if ts is not None:
            events[event_id]["timestamps"].append(ts)

    # Final summary
    summary = {}
    for eid, stats in events.items():
        total_frames = stats["smoke"] + stats["nosmoke"]
        smoke_ratio = stats["smoke"] / max(1, total_frames)

        if stats["timestamps"]:
            first_ts = min(stats["timestamps"])
            last_ts = max(stats["timestamps"])
        else:
            first_ts, last_ts = None, None

        summary[eid] = {
            "event_id": eid,
            "total_frames": total_frames,
            "smoke_frames": stats["smoke"],
            "nosmoke_frames": stats["nosmoke"],
            "smoke_ratio": round(smoke_ratio, 3),
            "first_timestamp": first_ts.isoformat() if first_ts else None,
            "last_timestamp": last_ts.isoformat() if last_ts else None,
            "images": sorted(stats["images"]),
        }

    return summary


def export_summary(summary: Dict[str, Dict], out_json: str, out_csv: str) -> None:
    """Export the event summary to JSON + CSV."""

    # JSON export
    with open(out_json, "w") as f:
        json.dump(summary, f, indent=2)

    # CSV export
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "event_id",
                "total_frames",
                "smoke_frames",
                "nosmoke_frames",
                "smoke_ratio",
                "first_timestamp",
                "last_timestamp",
            ]
        )
        for eid, s in summary.items():
            writer.writerow(
                [
                    eid,
                    s["total_frames"],
                    s["smoke_frames"],
                    s["nosmoke_frames"],
                    s["smoke_ratio"],
                    s["first_timestamp"],
                    s["last_timestamp"],
                ]
            )


if __name__ == "__main__":
    IMG_DIR = "path/to/images"
    LABEL_DIR = "path/to/labels"

    OUT_JSON = r"path\to\output\json"
    OUT_CSV = r"path\to\output\csv"

    summary = collect_event_statistics(IMG_DIR, LABEL_DIR)
    export_summary(summary, OUT_JSON, OUT_CSV)

    print(f"Collected stats for {len(summary)} events.")
    print(f"Saved: {OUT_JSON}")
    print(f"Saved: {OUT_CSV}")


Collected stats for 40 events.
Saved: D:\temp\event_summary.json
Saved: D:\temp\event_summary.csv


In [ ]:
# ----------------
# Figure 29-30
# ----------------

# ------------------------------------------------------------------
# Load event summary (JSON produced by the previous script)
# ------------------------------------------------------------------
SUMMARY_JSON = r"path\to\summary\json"

with open(SUMMARY_JSON, "r") as f:
    event_data = json.load(f)

# Extract statistics
event_ids = list(event_data.keys())
total_frames = [event_data[e]["total_frames"] for e in event_ids]
smoke_counts = [event_data[e]["smoke_frames"] for e in event_ids]
nosmoke_counts = [event_data[e]["nosmoke_frames"] for e in event_ids]

# ------------------------------------------------------------------
# Figure 29 – Distribution of Frames per Event
# ------------------------------------------------------------------
plt.figure(figsize=(6, 4))
plt.hist(total_frames, bins=20, edgecolor="black")
plt.title("Distribution of Frames per Event in Pyro-SDIS")
plt.xlabel("Number of Frames per Event")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(r"output\fig29.png")
plt.close()

# ------------------------------------------------------------------
# Figure 30 – Smoke vs. Non-Smoke Distribution per Event
# ------------------------------------------------------------------
x = np.arange(len(event_ids))
width = 0.38

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, smoke_counts, width=width, label="Smoke Frames", color="#7b9fc7")
plt.bar(x + width/2, nosmoke_counts, width=width, label="Non-Smoke Frames", color="#c77b7b")

plt.xlabel("Event ID")
plt.ylabel("Number of Frames")

plt.xticks(x, event_ids, rotation=70, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig(r"output\fig30.png")
plt.close()

print("Figures generated")


Figures generated


In [ ]:
# ----------------
# Figures 31-32-33
# ----------------

import json
import numpy as np
import os
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns


SAVE_DIR = r"path\to\save\dir"
os.makedirs(SAVE_DIR, exist_ok=True)

AREA_THRESHOLDS = {"small": 32**2, "medium": 96**2}
JSON_PATH = r"path\to\pyro\json"


# ======== Load + Stats ======== #

def load_coco(path: str) -> dict:
    """Load COCO annotation file."""
    with open(path, "r") as f:
        return json.load(f)


def compute_bbox_stats(coco: dict) -> list[dict]:
    """Compute bounding box statistics."""
    bboxes = []
    for ann in coco["annotations"]:
        x, y, w, h = ann["bbox"]
        area = w * h
        aspect = w / h if h > 0 else 0
        bboxes.append(
            {
                "image_id": ann["image_id"],
                "bbox": [x, y, w, h],
                "area": area,
                "aspect_ratio": aspect,
            }
        )
    return bboxes


def categorize_area(a: float) -> str:
    """Categorize bbox area into small/medium/large."""
    if a < AREA_THRESHOLDS["small"]:
        return "small"
    elif a < AREA_THRESHOLDS["medium"]:
        return "medium"
    return "large"


def iou(b1: list[float], b2: list[float]) -> float:
    """Compute IoU between two bounding boxes."""
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    xa, ya = max(x1, x2), max(y1, y2)
    xb, yb = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
    inter = max(0, xb - xa) * max(0, yb - ya)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0


# ======== Visualization ======== #


def plot_counts(counts: Counter) -> None:
    """Horizontal bar plot of box count frequency."""
    freq = Counter(counts.values())
    keys = sorted(freq.keys())
    vals = [freq[k] for k in keys]

    plt.figure(figsize=(8, 4))
    sns.barplot(x=vals, y=[str(k) for k in keys], palette=["#1e90ff"])
    plt.xlabel("Number of Images")
    plt.ylabel("Bounding Boxes per Image")
    plt.title("Distribution of Bounding Box Counts per Image")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/fig31.png", dpi=200)
    plt.close()


def plot_area_distribution(areas: np.ndarray) -> None:
    """KDE + Histogram of bbox areas."""
    log_areas = np.log10(areas)

    plt.figure(figsize=(8, 4))
    sns.histplot(log_areas, bins=40, kde=True, color="#b22222")
    plt.xlabel(r"$\log_{10}(Area\ [px^2])$")
    plt.title("Bounding Box Area Distribution")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/fig33.png", dpi=200)
    plt.close()


def plot_area_categories(categories: list[str]) -> None:
    """Small/Medium/Large pie chart with legend."""
    counts = Counter(categories)
    labels = [f"{k} ({counts[k]})" for k in counts]
    vals = list(counts.values())
    colors = ["#ff9999", "#66b3ff", "#99ff99"]

    plt.figure(figsize=(7, 6))

    wedges, texts, autotexts = plt.pie(
        vals,
        labels=labels,
        autopct="%1.1f%%",
        startangle=90,
        colors=colors,
        textprops={"fontsize": 12},
    )

    plt.title("COCO Area Categories of Smoke Bounding Boxes", fontsize=14)

    # --- Legend explaining the area rules --- #
    legend_labels = [
        "Small: area < 32² px",
        "Medium: 32² ≤ area < 96² px",
        "Large: area ≥ 96² px",
    ]

    plt.legend(
        wedges,
        legend_labels,
        title="Area Criteria",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.12),
        fontsize=11,
        title_fontsize=12,
        frameon=False,
    )

    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/fig32.png", dpi=200, bbox_inches="tight")
    plt.close()


# ======== MAIN ======== #


def run_analysis() -> None:
    """Run the full bbox analysis pipeline."""
    coco = load_coco(JSON_PATH)
    bboxes = compute_bbox_stats(coco)

    counts = Counter(b["image_id"] for b in bboxes)
    areas = np.array([b["area"] for b in bboxes])
    aspects = np.array([b["aspect_ratio"] for b in bboxes])
    categories = [categorize_area(a) for a in areas]

    # ==== PLOTS ==== #
    plot_counts(counts)
    plot_area_distribution(areas)
    plot_area_categories(categories)

    # ==== Sample + edge-case visuals ==== #
    sample_ids = list(counts.keys())[:3]
    extreme_small = [b for b in bboxes if b["area"] < 10]
    extreme_large = [b for b in bboxes if b["area"] > 1e6]
    edge_ids = [b["image_id"] for b in extreme_small[:2] + extreme_large[:2]]

    print("\n--- SUMMARY ---")
    print("Area categories:", Counter(categories))
    print("Extreme small boxes:", len(extreme_small))
    print("Extreme large boxes:", len(extreme_large))


if __name__ == "__main__":
    run_analysis()


C:\Users\HT\AppData\Local\Temp\ipykernel_22492\3302115955.py:58: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=vals, y=[str(k) for k in keys], palette=["#1e90ff"])
C:\Users\HT\AppData\Local\Temp\ipykernel_22492\3302115955.py:58: UserWarning: 
The palette list has fewer values (1) than needed (5) and will cycle, which may produce an uninterpretable plot.
  sns.barplot(x=vals, y=[str(k) for k in keys], palette=["#1e90ff"])



--- SUMMARY ---
Area categories: Counter({'medium': 16418, 'small': 13550, 'large': 2141})
Extreme small boxes: 0
Extreme large boxes: 0
